# Micrometeorology: Fusão de Dados de Sensores (.dat)

Neste notebook, testaremos a biblioteca interna `micrometeorology.sensors` para ingestão, concatenação e visualização rápida de arquivos `.dat` provenientes de dataloggers (ex: Campbell Scientific).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from micrometeorology.sensors.ingestion import merge_dat_files

plt.style.use("seaborn-v0_8-whitegrid")

## 1. Localização dos Arquivos DAT
Substitua a variável `RAW_DATA_DIR` abaixo pela pasta onde estão os arquivos brutos baixados da estação meteorológica.

In [ ]:
RAW_DATA_DIR = Path("../../data/raw/station_01")
file_list = list(RAW_DATA_DIR.glob("*.dat"))

print(f"Encontrados {len(file_list)} arquivos .dat na pasta.")

## 2. Fusão de vários arquivos
A função `merge_dat_files` lê cada `.dat` com `read_campbell_dat` e resolve a sobreposição POR COLUNA: numa mesma marca de tempo vale o primeiro valor não nulo na ordem cronológica dos arquivos, de modo que uma coluna que só existe no arquivo mais novo sobrevive mesmo quando o mais antigo cobre aquele instante. Não há PyArrow no caminho — é pandas.

In [ ]:
# O merge_dat_files fará todo o alinhamento temporal pra você!
if file_list:
    df_merged = merge_dat_files(file_list)
    print(f"Dataframe fundido com sucesso. Shape final: {df_merged.shape}")
    display(df_merged.head())
else:
    print("Nenhum arquivo .dat para simular. Verifique a variável RAW_DATA_DIR.")

## 3. Qualidade de Dados (Plot de Teste)
Sempre bom verificar se a variável principal (como Radiação ou Temperatura) não possui saltos malucos após a junção dos .dats.

In [ ]:
# Substitua 'rad_sw_in' pelo nome da coluna que deseja inspecionar
col_name = "rad_sw_in"

if file_list and col_name in df_merged.columns:
    plt.figure(figsize=(12, 5))
    plt.plot(df_merged.index, df_merged[col_name], label=col_name, color="orange")
    plt.title(f"Série Temporal Ininterrupta: {col_name}")
    plt.ylabel("Valor")
    plt.xlabel("Data/Hora")
    plt.legend()
    plt.tight_layout()
    plt.show()